In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('05_guiana_WDI.csv')
df = df.drop(columns=['Country Code', 'Series Code'])
df = df.dropna()

df_long = pd.melt(
    df,
    id_vars=["Country Name", "Series Name"],
    var_name="Year",
    value_name="Value"
)

# Limpeza opcional: extrair apenas o ano da coluna "Year"
df_long["Year"] = df_long["Year"].str.extract(r"(\d{4})")

# Converter valores para float (coerção de erros)
df_long["Value"] = pd.to_numeric(df_long["Value"], errors="coerce")

df_pivot = df_long.pivot_table(
    index=["Country Name", "Year"],
    columns="Series Name",
    values="Value"
).reset_index()

# Renomear colunas se necessário
df_pivot.columns.name = None  # Remove o nome do índice das colunas

# Visualizar resultado
df_pivot = df_pivot.rename(columns={
    'Country Name': 'Country',
    'Year': 'Index Year'
})
df_pivot

,Country,Index Year,Current health expenditure (% of GDP),GDP growth (annual %),Political Stability and Absence of Violence/Terrorism: Estimate
0,Argentina,2014,9.671300,-2.512615,-0.005122
1,Argentina,2015,10.229337,2.731160,0.008764
2,Argentina,2016,9.855589,-2.080328,0.197750
3,Argentina,2017,10.231748,2.818503,0.162542
4,Argentina,2018,10.291469,-2.617396,0.006910
...,...,...,...,...,...
105,Uruguay,2019,9.390277,0.928351,1.028297
106,Uruguay,2020,9.136885,-7.380142,1.054964
107,Uruguay,2021,9.359206,5.561893,1.045972
108,Uruguay,2022,NaN,4.707258,1.085830


In [2]:
df_2 = pd.read_csv('02_economic_freedom_index.csv', skiprows=4)
df_2 

,Country,Index Year,Overall Score,Property Rights,Government Integrity,Judicial Effectiveness,Tax Burden,Government Spending,Fiscal Health,Business Freedom,Labor Freedom,Monetary Freedom,Trade Freedom,Investment Freedom,Financial Freedom
0,Argentina,2023,51.0,34.6,56.8,42.8,66.9,53.9,33.6,55.5,55.1,36.5,61.2,55,60
1,Argentina,2022,50.1,35.1,45.1,57.9,73.3,53.0,16.8,55.1,51.0,37.9,60.6,55,60
2,Argentina,2021,52.7,46.1,54.0,45.7,70.4,52.8,38.4,59.5,46.3,41.9,62.6,55,60
3,Argentina,2020,53.1,50.5,49.7,47.0,69.6,50.7,24.7,60.2,46.5,53.7,69.2,55,60
4,Argentina,2019,52.2,47.8,33.5,44.5,69.3,49.5,33.0,56.4,46.9,60.2,70.0,55,60
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,Venezuela,2018,25.2,5.2,7.5,13.8,72.5,57.3,18.4,35.4,24.2,0.0,58.7,0,10
316,Venezuela,2017,27.0,6.8,11.6,10.3,72.5,51.5,15.2,39.7,28.5,16.8,60.7,0,10
317,Venezuela,2016,33.7,5.0,19.0,NaN,74.9,56.7,NaN,45.3,29.5,33.8,63.2,0,10
318,Venezuela,2015,34.3,5.0,20.0,NaN,75.0,52.0,NaN,41.6,24.2,42.8,62.8,0,20


In [3]:
df_pivot['Index Year'] = df_pivot['Index Year'].astype(int)
df_2['Index Year'] = df_2['Index Year'].astype(int)

df_combined = pd.merge(df_pivot, df_2, on=['Country', 'Index Year'], how='inner')
df_combined

,Country,Index Year,Current health expenditure (% of GDP),GDP growth (annual %),Political Stability and Absence of Violence/Terrorism: Estimate,Overall Score,Property Rights,Government Integrity,Judicial Effectiveness,Tax Burden,Government Spending,Fiscal Health,Business Freedom,Labor Freedom,Monetary Freedom,Trade Freedom,Investment Freedom,Financial Freedom
0,Argentina,2014,9.671300,-2.512615,-0.005122,44.6,15.0,29.5,NaN,63.5,49.9,NaN,53.9,44.9,60.0,68.9,30,30
1,Argentina,2015,10.229337,2.731160,0.008764,44.1,15.0,34.0,NaN,66.8,41.2,NaN,52.8,43.3,59.6,68.8,30,30
2,Argentina,2016,9.855589,-2.080328,0.197750,43.8,15.0,34.0,NaN,66.1,51.3,NaN,56.0,43.9,44.0,67.4,30,30
3,Argentina,2017,10.231748,2.818503,0.162542,50.4,32.4,38.2,39.6,62.6,54.6,56.4,57.3,46.1,50.9,66.7,50,50
4,Argentina,2018,10.291469,-2.617396,0.006910,52.3,40.8,32.6,44.5,65.7,55.6,52.6,56.2,43.3,51.3,70.3,55,60
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105,Uruguay,2019,9.390277,0.928351,1.028297,68.6,68.3,69.2,58.9,77.2,67.5,69.9,74.3,71.9,72.9,78.6,85,30
106,Uruguay,2020,9.136885,-7.380142,1.054964,69.1,72.0,73.5,59.8,75.2,66.5,72.3,73.6,71.0,72.7,77.4,85,30
107,Uruguay,2021,9.359206,5.561893,1.045972,69.3,68.1,75.7,66.8,72.6,66.9,80.5,72.3,70.5,72.5,70.8,85,30
108,Uruguay,2022,NaN,4.707258,1.085830,70.0,84.0,75.2,80.4,72.4,70.2,75.2,75.7,60.3,70.9,70.4,75,30


In [4]:
df_3 = pd.read_csv('03_corruption_perception.csv')
df_3 = df_3.drop(columns=['Economy ISO3', 'Indicator ID', 'Attribute 1', 'Attribute 2', 'Attribute 3', 'Partner', '2012', '2013'])

df_3 = df_3[~df_3['Indicator'].isin([
    'Corruption Perceptions Index Standard Error',
    'Corruption Perceptions Index Sources'
])]

df_long = pd.melt(
    df_3,
    id_vars=['Economy Name', 'Indicator'],
    var_name='Year',
    value_name='Value'
)

# 3. Converte o ano para inteiro (caso esteja como string)
df_long['Year'] = df_long['Year'].astype(int)

# 4. Pivot para colocar 'Rank' e 'Sources' como colunas
df_final = df_long.pivot_table(
    index=['Economy Name', 'Year'],
    columns='Indicator',
    values='Value'
).reset_index()

# 5. Renomeia as colunas para algo mais limpo (opcional)
df_final.columns.name = None
df_final = df_final.rename(columns={
    'Economy Name': 'Country',
    'Corruption Perceptions Index Rank': 'CPI Rank',
    'Corruption Perceptions Index Score': 'CPI Score'
})


df_final = df_final.rename(columns={
    'Year': 'Index Year'
})
df_final

,Country,Index Year,CPI Rank,CPI Score
0,Afghanistan,2014,172.0,12.0
1,Afghanistan,2015,166.0,11.0
2,Afghanistan,2016,169.0,15.0
3,Afghanistan,2017,177.0,15.0
4,Afghanistan,2018,172.0,16.0
...,...,...,...,...
1775,Zimbabwe,2019,158.0,24.0
1776,Zimbabwe,2020,157.0,24.0
1777,Zimbabwe,2021,157.0,23.0
1778,Zimbabwe,2022,157.0,23.0


In [5]:
df_combined_2 = pd.merge(df_combined, df_final, on=['Country', 'Index Year'], how='inner')
df_combined_2 = df_combined_2.rename(columns={
    'Current health expenditure (% of GDP)':'health',
    'GDP growth (annual %)':'g_GDP',
    'Political Stability and Absence of Violence/Terrorism: Estimate':'stability',
    'Overall Score':'scofree',
    'Property Rights':'propr',
    'Government Integrity':'govint',
    'Judicial Effectiveness':'judeff',
    'Tax Burden':'taxb',
    'Government Spending':'govspend',
    'Fiscal Health':'fischealth',
    'Business Freedom':'busfree',
    'Labor Freedom':'labfree',
    'Monetary Freedom':'monfree',
    'Trade Freedom':'tradfree',
    'Investment Freedom':'investfree',
    'Financial Freedom':'finfree',
    'CPI Rank':'CPIrank',
    'CPI Score':'CPIscore'
})
df_combined_2 

,Country,Index Year,health,g_GDP,stability,scofree,propr,govint,judeff,taxb,govspend,fischealth,busfree,labfree,monfree,tradfree,investfree,finfree,CPIrank,CPIscore
0,Argentina,2014,9.671300,-2.512615,-0.005122,44.6,15.0,29.5,NaN,63.5,49.9,NaN,53.9,44.9,60.0,68.9,30,30,107.0,34.0
1,Argentina,2015,10.229337,2.731160,0.008764,44.1,15.0,34.0,NaN,66.8,41.2,NaN,52.8,43.3,59.6,68.8,30,30,107.0,32.0
2,Argentina,2016,9.855589,-2.080328,0.197750,43.8,15.0,34.0,NaN,66.1,51.3,NaN,56.0,43.9,44.0,67.4,30,30,95.0,36.0
3,Argentina,2017,10.231748,2.818503,0.162542,50.4,32.4,38.2,39.6,62.6,54.6,56.4,57.3,46.1,50.9,66.7,50,50,85.0,39.0
4,Argentina,2018,10.291469,-2.617396,0.006910,52.3,40.8,32.6,44.5,65.7,55.6,52.6,56.2,43.3,51.3,70.3,55,60,85.0,40.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105,Uruguay,2019,9.390277,0.928351,1.028297,68.6,68.3,69.2,58.9,77.2,67.5,69.9,74.3,71.9,72.9,78.6,85,30,21.0,71.0
106,Uruguay,2020,9.136885,-7.380142,1.054964,69.1,72.0,73.5,59.8,75.2,66.5,72.3,73.6,71.0,72.7,77.4,85,30,21.0,71.0
107,Uruguay,2021,9.359206,5.561893,1.045972,69.3,68.1,75.7,66.8,72.6,66.9,80.5,72.3,70.5,72.5,70.8,85,30,18.0,73.0
108,Uruguay,2022,NaN,4.707258,1.085830,70.0,84.0,75.2,80.4,72.4,70.2,75.2,75.7,60.3,70.9,70.4,75,30,14.0,74.0


In [6]:
df_combined_2.to_csv('06_df_compilado_guiana.csv')